# GTEx v8 Junction Reads — Data Exploration & Aggregation

This notebook explores the GTEx v8 STAR junction read count data and walks through
the aggregation pipeline that transforms raw per-sample counts into per-tissue
summaries for our JunctionModality.

## Background

**What are junction reads?** When RNA-seq reads span an exon-exon boundary,
the aligner (STAR) splits them across the intron. Each unique
(chrom, donor_position, acceptor_position) pair is a **junction**, and the number
of reads supporting it is the evidence strength.

```
Genomic DNA:   ===EXON1===----intron----===EXON2===

Split read:         ACGTACGT              TGCATGCA
CIGAR:              8M  18N  8M
                        ^^^^                        
                    skipped intron (N operation)
```

## Data Sources

- **GTEx v8 STAR junctions**: `GTEx_Analysis_2017-06-05_v8_STARv2.5.3a_junctions.gct.gz` (4.25 GB)
  - 357,746 junctions × 17,382 samples
  - GCT format (tab-delimited matrix)
  - Genome build: GRCh38/hg38
- **Sample metadata**: `GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt` (11 MB)
  - Maps sample IDs → tissue names (55 tissue types)

In [ ]:
import gzip
from pathlib import Path

import numpy as np
import polars as pl

RAW_DIR = Path("../../data/mane/GRCh38/junction_data/raw")
OUTPUT_DIR = Path("../../data/mane/GRCh38/junction_data")

GCT_PATH = RAW_DIR / "GTEx_Analysis_2017-06-05_v8_STARv2.5.3a_junctions.gct.gz"
META_PATH = RAW_DIR / "GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt"

print(f"GCT file: {GCT_PATH.stat().st_size / 1e9:.2f} GB")
print(f"Metadata: {META_PATH.stat().st_size / 1e6:.1f} MB")

## 1. Sample Metadata — Mapping Samples to Tissues

GTEx v8 has ~23K total samples (RNA-seq, WGS, etc.), of which 17,382 have
junction data in the GCT file. Each sample comes from one of 55 tissue types.

In [ ]:
meta = pl.read_csv(META_PATH, separator="\t", infer_schema_length=10000)
print(f"Total samples in metadata: {meta.height:,}")
print(f"Columns: {meta.columns[:10]}")
meta.head(3)

In [ ]:
# Key columns: SAMPID (sample ID), SMTS (broad tissue), SMTSD (detailed tissue)
tissue_counts = (
    meta.group_by("SMTSD")
    .agg(pl.len().alias("n_samples"))
    .sort("n_samples", descending=True)
)
print(f"Tissue types: {tissue_counts.height}")
print(f"\nTop 10 tissues by sample count:")
tissue_counts.head(10)

In [ ]:
# Build sample → tissue mapping (only what we need)
sample_to_tissue = dict(zip(
    meta["SAMPID"].to_list(),
    meta["SMTSD"].to_list()
))
print(f"Sample→tissue mapping: {len(sample_to_tissue):,} entries")
print(f"Unique tissues: {len(set(sample_to_tissue.values()))}")

## 2. GCT File Structure — Raw Junction Counts

The GCT (Gene Cluster Text) format:
- Line 1: Version (`#1.2`)
- Line 2: Dimensions (`n_junctions  n_samples`)
- Line 3: Header (`Name  Description  SAMPLE1  SAMPLE2  ...`)
- Data rows: `chr1_12058_12178  ENSG00000223972.5  0  0  3  ...`

Each junction ID encodes `chrom_donor_acceptor` (1-based coordinates).
The Description column contains the Ensembl gene ID.

In [ ]:
# Read just the header and first 20 junctions to understand the structure
with gzip.open(GCT_PATH, "rt") as f:
    version = f.readline().strip()
    dims = f.readline().strip().split("\t")
    header = f.readline().strip().split("\t")
    sample_ids = header[2:]  # skip Name, Description
    
    print(f"GCT version: {version}")
    print(f"Dimensions: {dims[0]} junctions × {dims[1]} samples")
    print(f"Header columns: Name, Description, + {len(sample_ids)} sample IDs")
    print(f"First 3 sample IDs: {sample_ids[:3]}")
    print()
    
    # Read first 20 junction rows
    sample_rows = []
    for i in range(20):
        parts = f.readline().strip().split("\t")
        junction_id = parts[0]
        gene_id = parts[1]
        counts = np.array(parts[2:], dtype=np.int32)
        sample_rows.append({
            "junction_id": junction_id,
            "gene_id": gene_id,
            "total_reads": int(counts.sum()),
            "n_nonzero_samples": int(np.count_nonzero(counts)),
            "max_reads_any_sample": int(counts.max()),
        })

pl.DataFrame(sample_rows)

### Parsing Junction IDs

Each junction ID has the format `chrom_start_end` where:
- `chrom`: chromosome (e.g., chr1, chrX)
- `start`: donor position (5' splice site, end of upstream exon)
- `end`: acceptor position (3' splice site, start of downstream exon)

In [ ]:
def parse_junction_id(junction_id: str) -> tuple[str, int, int]:
    """Parse 'chr1_12058_12178' → ('chr1', 12058, 12178)."""
    parts = junction_id.split("_")
    end = int(parts[-1])
    start = int(parts[-2])
    chrom = "_".join(parts[:-2])
    return chrom, start, end

# Demo
for jid in ["chr1_12058_12178", "chr1_12722_13220", "chrX_100_500"]:
    chrom, start, end = parse_junction_id(jid)
    intron_len = end - start
    print(f"{jid} → chrom={chrom}, donor={start}, acceptor={end}, intron={intron_len}bp")

## 3. Sparsity Analysis

Most junctions have very few supporting reads in most samples. Understanding
the sparsity pattern tells us how to efficiently aggregate.

In [ ]:
# Sample 1000 random junctions to characterize the distribution
# (reading the full 357K × 17K matrix would be ~50 GB uncompressed)
np.random.seed(42)
n_total = 357746
n_sample = 1000
sample_indices = set(np.random.choice(n_total, n_sample, replace=False))

stats = []
with gzip.open(GCT_PATH, "rt") as f:
    f.readline()  # version
    f.readline()  # dims
    f.readline()  # header
    
    for idx, line in enumerate(f):
        if idx not in sample_indices:
            continue
        parts = line.rstrip("\n").split("\t")
        counts = np.array(parts[2:], dtype=np.int32)
        stats.append({
            "junction_id": parts[0],
            "total_reads": int(counts.sum()),
            "n_nonzero": int(np.count_nonzero(counts)),
            "max_in_sample": int(counts.max()),
            "sparsity": 1.0 - np.count_nonzero(counts) / len(counts),
        })
        if len(stats) >= n_sample:
            break

stats_df = pl.DataFrame(stats)
print(f"Sampled {stats_df.height} junctions")
print(f"\nSparsity (fraction of zero entries):")
print(f"  Mean:   {stats_df['sparsity'].mean():.3f}")
print(f"  Median: {stats_df['sparsity'].median():.3f}")
print(f"  Min:    {stats_df['sparsity'].min():.3f}")
print(f"\nRead count distribution:")
print(f"  Median total reads:    {stats_df['total_reads'].median():,.0f}")
print(f"  Mean total reads:      {stats_df['total_reads'].mean():,.0f}")
print(f"  Median nonzero samples: {stats_df['n_nonzero'].median():.0f} / 17382")
print(f"  Mean nonzero samples:   {stats_df['n_nonzero'].mean():.0f} / 17382")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Sparsity distribution
axes[0].hist(stats_df["sparsity"].to_numpy(), bins=50, edgecolor="black", alpha=0.7)
axes[0].set_xlabel("Sparsity (fraction of zero samples)")
axes[0].set_ylabel("Count")
axes[0].set_title("Junction Sparsity")
axes[0].axvline(stats_df["sparsity"].median(), color="red", linestyle="--", label="median")
axes[0].legend()

# Log total reads distribution
log_reads = np.log10(stats_df.filter(pl.col("total_reads") > 0)["total_reads"].to_numpy())
axes[1].hist(log_reads, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
axes[1].set_xlabel("log10(total reads)")
axes[1].set_ylabel("Count")
axes[1].set_title("Read Count Distribution")

# Nonzero samples distribution
axes[2].hist(stats_df["n_nonzero"].to_numpy(), bins=50, edgecolor="black", alpha=0.7, color="coral")
axes[2].set_xlabel("Number of samples with ≥1 read")
axes[2].set_ylabel("Count")
axes[2].set_title("Sample Support")

plt.tight_layout()
plt.show()

## 4. Per-Tissue Aggregation

The aggregation pipeline (in `scripts/aggregate_gtex_junctions.py`) works as follows:

1. **Build tissue index**: Map each of the 17,382 sample columns to one of 55 tissues
2. **Stream junctions**: For each of the 357,746 junctions:
   - Parse the row into a numpy array of counts
   - For each tissue, slice the relevant sample columns
   - Compute `total_reads = sum(counts)` and `n_samples = count(counts > 0)`
   - Only store non-zero tissue entries (sparse)
3. **Summarize**: Collapse tissue-level data into per-junction summary stats

This streaming approach avoids loading the full 357K × 17K matrix into memory.

In [ ]:
# Demonstrate the tissue-index approach on the first 100 junctions
# This is the core logic of the aggregation script

# Build tissue→column indices mapping
with gzip.open(GCT_PATH, "rt") as f:
    f.readline(); f.readline()
    header = f.readline().strip().split("\t")
    sample_ids = header[2:]

tissue_to_indices: dict[str, np.ndarray] = {}
for i, sid in enumerate(sample_ids):
    tissue = sample_to_tissue.get(sid)
    if tissue:
        tissue_to_indices.setdefault(tissue, []).append(i)

tissue_to_indices = {t: np.array(idxs) for t, idxs in tissue_to_indices.items()}

print(f"Tissues with samples: {len(tissue_to_indices)}")
print(f"\nSamples per tissue (top 5):")
for t in sorted(tissue_to_indices, key=lambda t: len(tissue_to_indices[t]), reverse=True)[:5]:
    print(f"  {len(tissue_to_indices[t]):5d}  {t}")

In [ ]:
# Aggregate first 100 junctions as demonstration
demo_rows = []
with gzip.open(GCT_PATH, "rt") as f:
    f.readline(); f.readline(); f.readline()  # skip header
    
    for i in range(100):
        parts = f.readline().rstrip("\n").split("\t")
        junction_id, gene_id = parts[0], parts[1]
        chrom, start, end = parse_junction_id(junction_id)
        counts = np.array(parts[2:], dtype=np.int32)
        
        for tissue, idxs in tissue_to_indices.items():
            tissue_counts = counts[idxs]
            total = int(tissue_counts.sum())
            n_supporting = int((tissue_counts > 0).sum())
            if total > 0:
                demo_rows.append({
                    "chrom": chrom, "start": start, "end": end,
                    "gene_id": gene_id, "tissue": tissue,
                    "total_reads": total, "n_samples": n_supporting,
                })

demo_tissue = pl.DataFrame(demo_rows)
print(f"100 junctions → {demo_tissue.height} tissue-level rows (non-zero only)")
print(f"Average tissues per junction: {demo_tissue.height / 100:.1f}")
demo_tissue.head(10)

In [ ]:
# Show one junction across tissues
example_junc = demo_tissue.filter(
    (pl.col("start") == demo_tissue["start"][3]) & 
    (pl.col("end") == demo_tissue["end"][3])
).sort("total_reads", descending=True)

print(f"Junction: {example_junc['chrom'][0]}:{example_junc['start'][0]}-{example_junc['end'][0]}")
print(f"Gene: {example_junc['gene_id'][0]}")
print(f"Tissues with support: {example_junc.height}")
print()
example_junc

## 5. Aggregated Output — Per-Junction Summary

The final output collapses tissue-level data into a single row per junction.
This is what the JunctionModality consumes.

Columns:
- `chrom`, `start`, `end`, `gene_id`: junction coordinates
- `max_reads`: highest total_reads in any tissue
- `sum_reads`: total reads across all tissues
- `n_tissues`: number of tissues with ≥1 supporting read
- `tissue_breadth`: n_tissues / total_tissues (0–1 scale)
- `max_tissue`: tissue with highest read count
- `mean_samples_per_tissue`: average number of samples per tissue with support

In [ ]:
# Load pre-computed aggregated output (from scripts/aggregate_gtex_junctions.py)
summary_path = OUTPUT_DIR / "junctions_gtex_v8.parquet"
tissue_path = OUTPUT_DIR / "junctions_gtex_v8_by_tissue.parquet"

if summary_path.exists():
    summary = pl.read_parquet(summary_path)
    print(f"Junction summary: {summary.height:,} junctions, {summary.width} columns")
    print(f"Columns: {summary.columns}")
    print(f"Chromosomes: {summary['chrom'].n_unique()}")
    summary.head(10)
else:
    print(f"Summary not yet generated. Run:")
    print(f"  python scripts/aggregate_gtex_junctions.py --tissue-level")

In [ ]:
if summary_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Tissue breadth distribution
    axes[0].hist(summary["tissue_breadth"].to_numpy(), bins=50, edgecolor="black", alpha=0.7)
    axes[0].set_xlabel("Tissue breadth (fraction of tissues)")
    axes[0].set_ylabel("Junctions")
    axes[0].set_title("Constitutive vs Tissue-Specific")
    axes[0].axvline(0.5, color="red", linestyle="--", alpha=0.5)

    # Log max reads
    log_max = np.log10(summary.filter(pl.col("max_reads") > 0)["max_reads"].to_numpy())
    axes[1].hist(log_max, bins=50, edgecolor="black", alpha=0.7, color="steelblue")
    axes[1].set_xlabel("log10(max reads in any tissue)")
    axes[1].set_ylabel("Junctions")
    axes[1].set_title("Evidence Strength")

    # N tissues distribution
    axes[2].hist(summary["n_tissues"].to_numpy(), bins=55, edgecolor="black", alpha=0.7, color="coral")
    axes[2].set_xlabel("Number of tissues with support")
    axes[2].set_ylabel("Junctions")
    axes[2].set_title("Tissue Support Distribution")

    plt.tight_layout()
    plt.show()

    # Summary stats
    n_broad = summary.filter(pl.col("n_tissues") >= 10).height
    n_rare = summary.filter(pl.col("n_tissues") <= 2).height
    n_single = summary.filter(pl.col("n_tissues") == 1).height
    print(f"\nJunction categories:")
    print(f"  Broad (≥10 tissues):  {n_broad:>8,} ({100*n_broad/summary.height:.1f}%)")
    print(f"  Rare  (≤2 tissues):   {n_rare:>8,} ({100*n_rare/summary.height:.1f}%)")
    print(f"  Single-tissue:        {n_single:>8,} ({100*n_single/summary.height:.1f}%)")

## 6. Connecting to the JunctionModality

The JunctionModality in `src/agentic_spliceai/splice_engine/features/modalities/junction.py`
takes this aggregated junction data and maps it to individual genomic positions.

For each position in our feature DataFrame:
1. Find all junctions where this position is a **donor** (= start) or **acceptor** (= end)
2. Compute features: `log1p_reads`, `has_support`, `n_partners`, `tissue_breadth`, `PSI`, etc.
3. Positions with no junction support get zeros (graceful degradation)

### Dual Role: Feature vs Label

Junction evidence serves different roles depending on the meta-layer model:

| Meta Model | Junction as... | Config |
|---|---|---|
| M1 (enhanced canonical) | **Feature** — confirms base model predictions | `full_stack.yaml` |
| M2 (alternative site) | **Feature** — tissue breadth & PSI indicate alternative splicing | `full_stack.yaml` |
| M3 (novel predictor) | **Held-out label** — "does this position have junction support?" | `meta_m3_novel.yaml` (junction excluded) |
| M4 (perturbation) | **Validation** — not in training | — |

In [ ]:
# Preview: how junction data maps to our analysis_sequences positions
# Load our chr22 feature data
analysis_path = Path("../../data/mane/GRCh38/openspliceai_eval/analysis_sequences/analysis_sequences_chr22.parquet")

if analysis_path.exists() and summary_path.exists():
    features = pl.read_parquet(analysis_path)
    chr22_junctions = summary.filter(pl.col("chrom") == "chr22")
    
    print(f"Feature positions (chr22): {features.height:,}")
    print(f"Junctions (chr22): {chr22_junctions.height:,}")
    
    # How many of our positions are junction donors or acceptors?
    donor_positions = set(chr22_junctions["start"].to_list())
    acceptor_positions = set(chr22_junctions["end"].to_list())
    our_positions = set(features["position"].to_list())
    
    donor_overlap = our_positions & donor_positions
    acceptor_overlap = our_positions & acceptor_positions
    any_overlap = donor_overlap | acceptor_overlap
    
    print(f"\nPositions with junction evidence:")
    print(f"  Donor matches:    {len(donor_overlap):,}")
    print(f"  Acceptor matches: {len(acceptor_overlap):,}")
    print(f"  Any match:        {len(any_overlap):,} ({100*len(any_overlap)/features.height:.1f}%)")
else:
    print("Prerequisite files not yet available.")
    if not analysis_path.exists():
        print(f"  Missing: {analysis_path}")
    if not summary_path.exists():
        print(f"  Missing: {summary_path}")

## Summary

**Pipeline**: Raw GTEx GCT (4.25 GB, 357K × 17K) → Stream aggregate by tissue → Per-junction summary parquet

**Key stats**:
- 357,746 junctions across 55 tissues
- Data is highly sparse (most junctions observed in only a few samples/tissues)
- Constitutive junctions (broad tissue support) vs tissue-specific (1-2 tissues)

**Next steps**:
1. Run `scripts/aggregate_gtex_junctions.py --tissue-level` to generate the full aggregation
2. Wire the output to `JunctionModality` via the feature registry
3. Uncomment junction in `full_stack.yaml` and regenerate features
4. Train meta-layer M2 (with junction features) and M3 (junction as label)